# Experiment: Role-Labeled To Whisper-Like Test

Objective:
- Convert a `role_labeled` interview JSON into a whisper-like test object for `openwillis.speech`.
- Drop every segment where `role == "unknown"`.
- Build synthetic `words` stubs so Whisper-like validation passes.
- Preserve useful metadata while making the object runnable for a structural test.


In [5]:
from __future__ import annotations

import copy
import json
import re
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(items, **kwargs):
        return items

PROJECT_ROOT = Path("/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest")
INPUT_JSON = Path("/Users/pelmeshek1706/Downloads/Telegram Desktop/translated_uk_gpt54_flex_full 4/role_labeled/305.json")
OUTPUT_JSON = PROJECT_ROOT / "tmp" / "role_labeled_305_whisper_like_stub.json"
INPUT_DIR = INPUT_JSON.parent
OUTPUT_DIR = PROJECT_ROOT / "tmp" / "role_labeled_whisper_like_stub_batch_ukr_26-03-2026"
OWS_SRC = PROJECT_ROOT / "openwillis" / "openwillis-speech" / "src"

TARGET_SPEAKERS = {
    "participant": "participant",
    "interviewer": "interviewer",
}
TURN_MODES = ["speaker", "segment"]
DROP_UNKNOWN = True

if str(OWS_SRC) not in sys.path:
    sys.path.insert(0, str(OWS_SRC))

INPUT_JSON, OUTPUT_JSON, INPUT_DIR, OUTPUT_DIR


(PosixPath('/Users/pelmeshek1706/Downloads/Telegram Desktop/translated_uk_gpt54_flex_full 4/role_labeled/305.json'),
 PosixPath('/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_305_whisper_like_stub.json'),
 PosixPath('/Users/pelmeshek1706/Downloads/Telegram Desktop/translated_uk_gpt54_flex_full 4/role_labeled'),
 PosixPath('/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026'))

## Notes

- The source file is **not** whisper-like: it has `role`, but no `speaker`, and no `segments[*].words`.
- The stub below is only for structural testing of the pipeline. Timing is distributed uniformly across tokens inside each segment.
- This is enough for format checks and smoke-tests, but not for scientifically valid pause/coherence metrics.


In [3]:
TOKEN_RE = re.compile(r"\S+")

def build_word_stub(text: str, start: float, end: float, speaker: str, score: float = 1.0) -> list[dict]:
    tokens = TOKEN_RE.findall((text or "").strip())
    if not tokens:
        return []

    start = float(start or 0.0)
    end = float(end if end is not None else start)
    if end < start:
        end = start

    duration = max(end - start, 0.0)
    step = duration / max(len(tokens), 1)
    words = []
    for idx, token in enumerate(tokens):
        w_start = start + idx * step
        w_end = end if idx == len(tokens) - 1 else start + (idx + 1) * step
        words.append({
            "word": token,
            "start": round(w_start, 3),
            "end": round(max(w_end, w_start), 3),
            "score": score,
            "speaker": speaker,
        })
    return words

def convert_role_labeled_to_whisper_like(data: dict, drop_unknown: bool = True) -> dict:
    source = copy.deepcopy(data)
    new_segments = []
    new_word_segments = []
    new_turn_decisions = []

    for seg in source.get("segments", []):
        role = seg.get("role")
        if drop_unknown and role == "unknown":
            continue

        speaker = role or "unknown"
        new_seg = copy.deepcopy(seg)
        new_seg["speaker"] = speaker
        new_seg["words"] = build_word_stub(
            text=new_seg.get("text", ""),
            start=new_seg.get("start", 0.0),
            end=new_seg.get("end", 0.0),
            speaker=speaker,
        )
        new_segments.append(new_seg)
        new_word_segments.extend(copy.deepcopy(new_seg["words"]))

    for turn in source.get("turn_decisions", []):
        role = turn.get("role")
        if drop_unknown and role == "unknown":
            continue
        new_turn = copy.deepcopy(turn)
        new_turn["speaker"] = role or "unknown"
        new_turn_decisions.append(new_turn)

    result = {
        key: value
        for key, value in source.items()
        if key not in {"segments", "word_segments", "turn_decisions", "text", "source_text_en"}
    }
    result["segments"] = new_segments
    result["word_segments"] = new_word_segments
    result["text"] = " ".join(seg.get("text", "") for seg in new_segments).strip()
    result["source_text_en"] = " ".join(seg.get("source_text_en", "") for seg in new_segments).strip()
    result["turn_decisions"] = new_turn_decisions
    return result

def convert_role_labeled_directory_to_whisper_like(
    input_dir: str | Path,
    output_dir: str | Path,
    drop_unknown: bool = True,
) -> list[dict]:
    """Convert every `*.json` in `input_dir` and write it to `output_dir` with the same file name."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    if not input_dir.is_dir():
        raise NotADirectoryError(f"Input path is not a directory: {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []
    input_paths = sorted(input_dir.glob("*.json"))

    for input_path in tqdm(input_paths, desc="role_labeled -> whisper_like", unit="file"):
        raw = json.loads(input_path.read_text(encoding="utf-8"))
        converted = convert_role_labeled_to_whisper_like(raw, drop_unknown=drop_unknown)
        output_path = output_dir / input_path.name
        output_path.write_text(json.dumps(converted, ensure_ascii=False, indent=2), encoding="utf-8")
        summary_rows.append({
            "file_name": input_path.name,
            "input_segments": len(raw.get("segments", [])),
            "output_segments": len(converted.get("segments", [])),
            "output_word_segments": len(converted.get("word_segments", [])),
            "output_path": str(output_path),
        })

    return summary_rows

def describe_structure(obj: dict) -> dict:
    segments = obj.get("segments", [])
    word_segments = obj.get("word_segments", [])
    return {
        "top_level_keys": list(obj.keys()),
        "n_segments": len(segments),
        "n_word_segments": len(word_segments),
        "roles": Counter(seg.get("role") for seg in segments),
        "speakers": Counter(seg.get("speaker") for seg in segments if seg.get("speaker") is not None),
        "segments_with_words": sum(1 for seg in segments if seg.get("words")),
    }


In [4]:
raw = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
converted = convert_role_labeled_to_whisper_like(raw, drop_unknown=DROP_UNKNOWN)

print("Before:")
pprint(describe_structure(raw))

print("After:")
pprint(describe_structure(converted))

print("Sample converted segment (without full words payload):")
sample_segment = {k: v for k, v in converted["segments"][0].items() if k != "words"}
pprint(sample_segment)
print("sample words:", converted["segments"][0]["words"][:5])


Before:
{'n_segments': 331,
 'n_word_segments': 0,
 'roles': Counter({'participant': 228, 'interviewer': 93, 'unknown': 10}),
 'segments_with_words': 0,
 'speakers': Counter(),
 'top_level_keys': ['segments',
                    'word_segments',
                    'text',
                    'source_text_en',
                    'translation_meta',
                    'source_cleanup_meta',
                    'turn_decisions']}
After:
{'n_segments': 321,
 'n_word_segments': 2993,
 'roles': Counter({'participant': 228, 'interviewer': 93}),
 'segments_with_words': 321,
 'speakers': Counter({'participant': 228, 'interviewer': 93}),
 'top_level_keys': ['translation_meta',
                    'source_cleanup_meta',
                    'segments',
                    'word_segments',
                    'text',
                    'source_text_en',
                    'turn_decisions']}
Sample converted segment (without full words payload):
{'decision_source': 'model_turn_pass',
 'end': 20

In [5]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(converted, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote: {OUTPUT_JSON}")
print(f"Segments kept: {len(converted['segments'])}")
print(f"Word stubs created: {len(converted['word_segments'])}")


Wrote: /Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_305_whisper_like_stub.json
Segments kept: 321
Word stubs created: 2993


In [6]:
batch_summary = convert_role_labeled_directory_to_whisper_like(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    drop_unknown=DROP_UNKNOWN,
)
print(f"Files written: {len(batch_summary)}")
pprint(batch_summary[:3])


role_labeled -> whisper_like: 100%|██████████| 275/275 [00:10<00:00, 26.77file/s]

Files written: 275
[{'file_name': '300.json',
  'input_segments': 140,
  'output_path': '/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026/300.json',
  'output_segments': 127,
  'output_word_segments': 520},
 {'file_name': '301.json',
  'input_segments': 197,
  'output_path': '/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026/301.json',
  'output_segments': 191,
  'output_word_segments': 1379},
 {'file_name': '302.json',
  'input_segments': 169,
  'output_path': '/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026/302.json',
  'output_segments': 161,
  'output_word_segments': 854}]


In [5]:
converted

{'translation_meta': {'method': 'openai_translate_uk',
  'prompt_version': 'translate_uk_v3',
  'model': 'gpt-5.4',
  'service_tier': 'flex',
  'target_lang': 'Ukrainian',
  'translate_roles': ['participant', 'interviewer'],
  'participant_id': '300',
  'participant_gender': 'male',
  'participant_gender_source': '/Users/poluidol/Documents/work/airest/airest_perp_research/data_analysis/datasets/old/dcwoz_eng_new_gemma_merged_full_updated_4-2-26.csv',
  'n_segments_before': 140,
  'n_segments_after': 140,
  'translated_segments': 127,
  'api_usage': {'input_tokens': 11885,
   'output_tokens': 4298,
   'total_tokens': 16183},
  'word_alignment': 'not_available_for_translated_text',
  'view_role': 'all_roles'},
 'source_cleanup_meta': {'method': 'openai_hybrid_role_cleanup',
  'prompt_version': 'hybrid_role_cleanup_v2',
  'turn_model': 'gpt-5.2',
  'word_model': 'gpt-5.2',
  'language_hint': 'en',
  'n_turns_before': 140,
  'n_role_spans_after': 140,
  'model_turn_pass_turns': 140,
  'wor

## Test `openwillis.speech`

The next cells run the same pipeline you discussed earlier, but against the converted test object.
Because the `words` are synthetic stubs, the output is useful for plumbing checks, not for real acoustic/linguistic interpretation.


In [6]:
import pandas as pd
import openwillis.speech as ows
from openwillis.speech.speech_attribute import filter_whisper, get_config
from openwillis.speech.util.speech import coherence

coherence.COHERENCE_BACKEND = "gemma"
measures = get_config(str((OWS_SRC / "openwillis" / "speech" / "speech_attribute.py").resolve()), "text.json")

utterance_views = {}
utterance_summary_rows = []
for turn_mode in TURN_MODES:
    _, utterances_i = filter_whisper(converted, measures, whisper_turn_mode=turn_mode)
    utterance_views[turn_mode] = utterances_i[[
        measures["speaker_label"],
        measures["utterance_ids"],
        measures["utterance_text"],
    ]].copy()
    utterance_summary_rows.append({
        "turn_mode": turn_mode,
        "num_utterances": len(utterances_i),
        "speaker_counts": utterances_i[measures["speaker_label"]].value_counts(dropna=False).to_dict(),
    })

utterance_summary = pd.DataFrame(utterance_summary_rows)
display(utterance_summary)
display(utterance_views["speaker"].head(12))
display(utterance_views["segment"].head(12))


/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,turn_mode,num_utterances,speaker_counts
0,speaker,75,"{'SPEAKER_A': 38, 'SPEAKER_B': 37}"
1,segment,127,"{'SPEAKER_A': 66, 'SPEAKER_B': 61}"


,speaker_label,utterance_ids,utterance_text
0,SPEAKER_A,"(0, 0)",Добре.
1,SPEAKER_B,"(1, 67)","Привіт, я Еллі. Дякую, що прийшли сьогодні. Ме..."
2,SPEAKER_A,"(68, 68)",Добре.
3,SPEAKER_B,"(69, 71)",Звідки ви родом?
4,SPEAKER_A,"(72, 73)","Атланта, Джорджія."
5,SPEAKER_B,"(74, 79)",Справді? Чому ви переїхали до Лос-Анджелеса?
6,SPEAKER_A,"(80, 82)",Мої батьки звідси.
7,SPEAKER_B,"(83, 85)",Як вам Лос-Анджелес?
8,SPEAKER_A,"(86, 88)",Мені дуже подобається.
9,SPEAKER_B,"(89, 94)",Що вам найбільше подобається в Лос-Анджелесі?


,speaker_label,utterance_ids,utterance_text
0,SPEAKER_A,"(0, 0)",Добре.
1,SPEAKER_B,"(1, 3)","Привіт, я Еллі."
2,SPEAKER_B,"(4, 7)","Дякую, що прийшли сьогодні."
3,SPEAKER_B,"(8, 18)","Мене створили, щоб розмовляти з людьми в безпе..."
4,SPEAKER_B,"(19, 22)",Сприймайте мене як друга.
5,SPEAKER_B,"(23, 25)",Я не засуджую.
6,SPEAKER_B,"(26, 27)",Не можу.
7,SPEAKER_B,"(28, 29)",Я комп'ютер.
8,SPEAKER_B,"(30, 43)","Я тут, щоб дізнаватися про людей, і мені б дуж..."
9,SPEAKER_B,"(44, 51)","Я поставлю кілька запитань, щоб ми могли почати."


In [7]:
TARGET_SPEAKERS.items()

dict_items([('participant', 'SPEAKER_A'), ('interviewer', 'SPEAKER_B')])

In [ ]:
comparison_rows = []
feature_outputs = {}
from openwillis.speech.util.speech import coherence

coherence.COHERENCE_BACKEND = "gemma"
for speaker_name, label in TARGET_SPEAKERS.items():
    for turn_mode in TURN_MODES:
        words_i, turns_i, summary_i = ows.speech_characteristics(
            json_conf=converted,
            option="coherence",
            language="ua",
            speaker_label=label,
            min_coherence_turn_length=2,
            whisper_turn_mode=turn_mode,
        )
        feature_outputs[(speaker_name, turn_mode)] = {
            "words": words_i,
            "turns": turns_i,
            "summary": summary_i,
        }
        summary_row = summary_i.iloc[0] if not summary_i.empty else {}
        comparison_rows.append({
            "speaker_name": speaker_name,
            "speaker_label": label,
            "turn_mode": turn_mode,
            "words_rows": len(words_i),
            "turn_rows": len(turns_i),
            "speaker_percentage": summary_row.get("speaker_percentage"),
            "speech_minutes": summary_row.get("speech_length_minutes"),
            "speech_words": summary_row.get("speech_length_words"),
            "mean_turn_words": summary_row.get("mean_turn_length_words"),
            "num_turns": summary_row.get("num_turns"),
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [9]:
feature_outputs.keys()

dict_keys([('participant', 'speaker'), ('participant', 'segment'), ('interviewer', 'speaker'), ('interviewer', 'segment')])

In [12]:
pd.set_option("display.max_columns", None)

In [13]:
display(feature_outputs[("participant", "segment")]["turns"].head(5))
display(feature_outputs[("interviewer", "segment")]["turns"].head(5))

,pre_turn_pause,turn_length_minutes,turn_length_words,words_per_min,syllables_per_min,speech_percentage,mean_pause_length,pause_variability,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,first_person_sentiment_positive,first_person_sentiment_negative,word_repeat_percentage,phrase_repeat_percentage,first_order_sentence_tangeniality,second_order_sentence_tangeniality,turn_to_turn_tangeniality,semantic_perplexity,semantic_perplexity_5,semantic_perplexity_11,semantic_perplexity_15,interrupt_flag,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader
0,NaN,0.004,1,250.0,500.0,100.0,NaN,NaN,0.3041,0.312854,0.383046,-0.00226,1.0,0.0,0.0,0.3034,1.0,1.0,1.0,1.0,1.0,0.0,30.410030,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,100.000,0.0
1,1.321,0.00735,1,136.054422,272.108844,100.0,NaN,NaN,0.3041,0.312854,0.383046,-0.00226,1.0,0.0,0.0,0.3034,1.0,1.0,1.0,1.0,1.0,0.0,30.410030,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,100.000,0.0
2,0.020,0.01335,2,149.812734,374.531835,100.0,0.0,0,0.076211,0.126782,0.797007,-0.013056,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,7.621057,0.000000,0.0,0.0,NaN,NaN,NaN,53.678970,38028.531250,2109.415771,325.966980,False,0.000,0.0
3,0.061,0.024033,3,124.82663,249.653259,100.0,0.0,0.0,0.181997,0.22029,0.597713,-0.009887,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,25.0,13.649806,5.507242,0.0,0.0,NaN,NaN,0.310309,192.641861,46608.953125,285.909668,139.441147,False,0.000,0.0
4,0.020,0.00735,3,408.163265,1224.489796,100.0,0.0,0.0,0.838787,0.025148,0.136065,0.205593,0.607,0.0,0.393,0.4754,1.0,1.0,1.0,1.0,1.0,25.0,62.909031,0.628689,0.0,0.0,NaN,NaN,0.478692,93.751266,189812.390625,154.441147,93.751625,False,45.525,0.0


,pre_turn_pause,turn_length_minutes,turn_length_words,words_per_min,syllables_per_min,speech_percentage,mean_pause_length,pause_variability,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,first_person_sentiment_positive,first_person_sentiment_negative,word_repeat_percentage,phrase_repeat_percentage,first_order_sentence_tangeniality,second_order_sentence_tangeniality,turn_to_turn_tangeniality,semantic_perplexity,semantic_perplexity_5,semantic_perplexity_11,semantic_perplexity_15,interrupt_flag,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader
0,12.163,0.0207,3,144.927536,241.545894,100.0,0.0,0.0,0.324004,0.031072,0.644924,0.075419,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,20.000000,25.920296,0.621438,0.0,0.0,NaN,NaN,NaN,230.141312,158548.968750,5593.322266,230.141754,False,0.000000,0.0
1,0.440,0.0177,4,225.988701,451.977401,100.0,0.0,0.0,0.86596,0.030542,0.103498,0.210854,0.457,0.0,0.543,0.3653,1.0,1.0,1.0,1.0,1.0,0.000000,86.596000,0.000000,0.0,0.0,NaN,NaN,0.541706,36.899506,51641.128906,543.133240,225.937531,False,45.700000,0.0
2,0.701,0.051067,11,215.4047,587.467363,100.0,0.0,0.0,0.510329,0.058403,0.431268,0.115901,0.371,0.0,0.629,0.6505,1.0,1.0,1.0,1.0,1.0,7.692308,47.107321,0.449252,0.0,0.0,NaN,NaN,0.273656,36.457592,186197.734375,5278.930664,2383.861328,False,34.246154,0.0
3,0.781,0.015683,4,255.047821,510.095643,100.0,0.0,0.0,0.053372,0.773464,0.173164,-0.182794,0.516,0.0,0.484,0.4939,1.0,1.0,1.0,1.0,1.0,20.000000,4.269771,15.469279,0.0,0.0,NaN,NaN,0.343111,1053.104736,7027.752441,6225.364258,1721.738647,False,41.280000,0.0
4,0.781,0.016017,3,187.304891,312.174818,100.0,0.0,0.0,0.193306,0.409793,0.396901,-0.055809,0.516,0.0,0.484,0.2811,1.0,1.0,1.0,1.0,1.0,25.000000,14.497969,10.244818,0.0,0.0,NaN,NaN,0.493148,238.304977,16893.980469,428.919128,238.305099,False,38.700000,0.0


In [9]:
import json

file_path = "/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026/300.json"

with open(file_path, "r", encoding="utf-8") as f:
    ukr_data = json.load(f)

ukr_data

{'translation_meta': {'method': 'openai_translate_uk',
  'prompt_version': 'translate_uk_v3',
  'model': 'gpt-5.4',
  'service_tier': 'flex',
  'target_lang': 'Ukrainian',
  'translate_roles': ['participant', 'interviewer'],
  'participant_id': '300',
  'participant_gender': 'male',
  'participant_gender_source': '/Users/poluidol/Documents/work/airest/airest_perp_research/data_analysis/datasets/old/dcwoz_eng_new_gemma_merged_full_updated_4-2-26.csv',
  'n_segments_before': 140,
  'n_segments_after': 140,
  'translated_segments': 127,
  'api_usage': {'input_tokens': 11885,
   'output_tokens': 4298,
   'total_tokens': 16183},
  'word_alignment': 'not_available_for_translated_text',
  'view_role': 'all_roles'},
 'source_cleanup_meta': {'method': 'openai_hybrid_role_cleanup',
  'prompt_version': 'hybrid_role_cleanup_v2',
  'turn_model': 'gpt-5.2',
  'word_model': 'gpt-5.2',
  'language_hint': 'en',
  'n_turns_before': 140,
  'n_role_spans_after': 140,
  'model_turn_pass_turns': 140,
  'wor

In [ ]:
from phonova import SpeechAnalyzer

analyzer_ukr = SpeechAnalyzer(language="ua", coherence_backend="gemma")

INFO:pymorphy3.opencorpora_dict.wrapper:Loading dictionaries from /Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/pymorphy3_dicts_uk/data
INFO:pymorphy3.opencorpora_dict.wrapper:format: 2.4, revision: 1, updated: 2022-09-13T18:45:24.998984


In [10]:
words_ukr, turns_ukr, summary_ukr = analyzer_ukr.analyze_transcript(
    json_conf=ukr_data,
    option="coherence",
    speaker_label="participant",
    min_coherence_turn_length=2,
    whisper_turn_mode="segment",
)

/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: 'ь'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: '—'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: 'ь'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: '—'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/fi

In [12]:
analyzer_eng = SpeechAnalyzer(language="en", coherence_backend="gemma")

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: google/embeddinggemma-300m
INFO:sentence_transformers.SentenceTransformer:14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']


In [ ]:
import json

file_path = "/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_eng_26-03-2026/300.json"

with open(file_path, "r", encoding="utf-8") as f:
    eng_data = json.load(f)

eng_data

In [13]:
words_eng, turns_eng, summary_eng = analyzer_eng.analyze_transcript(
    json_conf=eng_data,
    option="coherence",
    speaker_label="participant",
    min_coherence_turn_length=2,
    whisper_turn_mode="segment",
)

In [16]:
import pandas as pd
pd.set_option("display.max_columns", None)

In [17]:
display(summary_ukr)
display(summary_eng)

,file_length,speech_length_minutes,speech_length_words,words_per_min,syllables_per_min,mean_pre_word_pause,mean_pause_variability,speech_percentage,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,prop_verb_past,prop_function_words,first_person_sentiment_positive,first_person_sentiment_negative,first_person_sentiment_overall,word_repeat_percentage,phrase_repeat_percentage,word_coherence_mean,word_coherence_var,word_coherence_5_mean,word_coherence_5_var,word_coherence_10_mean,word_coherence_10_var,word_coherence_variability_2_mean,word_coherence_variability_2_var,word_coherence_variability_3_mean,word_coherence_variability_3_var,word_coherence_variability_4_mean,word_coherence_variability_4_var,word_coherence_variability_5_mean,word_coherence_variability_5_var,word_coherence_variability_6_mean,word_coherence_variability_6_var,word_coherence_variability_7_mean,word_coherence_variability_7_var,word_coherence_variability_8_mean,word_coherence_variability_8_var,word_coherence_variability_9_mean,word_coherence_variability_9_var,word_coherence_variability_10_mean,word_coherence_variability_10_var,num_turns,num_one_word_turns,mean_turn_length_minutes,mean_turn_length_words,mean_pre_turn_pause,speaker_percentage,first_order_sentence_tangeniality_mean,first_order_sentence_tangeniality_var,second_order_sentence_tangeniality_mean,second_order_sentence_tangeniality_var,turn_to_turn_tangeniality_mean,turn_to_turn_tangeniality_var,turn_to_turn_tangeniality_slope,semantic_perplexity_mean,semantic_perplexity_var,semantic_perplexity_5_mean,semantic_perplexity_5_var,semantic_perplexity_11_mean,semantic_perplexity_11_var,semantic_perplexity_15_mean,semantic_perplexity_15_var,num_interrupts,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader,first_person_sentiment_overall_vader
0,10.692533,5.850883,520,88.875469,191.253173,0.0,0.0,54.719337,0.280084,0.238961,0.480955,0.041122,0.308,0.041,0.651,0.9995,0.977143,0.936538,0.854337,0.776708,0.695395,5.865522,0.288136,0.356061,26.317564,1.361495,20.904131,0.405512,0.0,0.606044,0.008957,0.697155,0.003225,0.668208,0.001623,0.599774,0.010808,0.600044,0.010539,0.610938,0.012413,0.604197,0.009801,0.611355,0.008289,0.585886,0.010951,0.624852,0.016538,0.600263,0.006922,0.584019,0.013484,127,26,0.04607,4.094488,2.113921,26.730803,NaN,NaN,NaN,NaN,0.440573,0.011523,0.000203,712.684793,6.295328e+06,269890.043828,3.353436e+11,4372.1993,3.065547e+07,1463.982299,7.519573e+06,0,25.648716,0.28315,25.703991


,file_length,speech_length_minutes,speech_length_words,words_per_min,syllables_per_min,mean_pre_word_pause,mean_pause_variability,speech_percentage,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,prop_verb_past,prop_function_words,first_person_sentiment_positive,first_person_sentiment_negative,first_person_sentiment_overall,word_repeat_percentage,phrase_repeat_percentage,word_coherence_mean,word_coherence_var,word_coherence_5_mean,word_coherence_5_var,word_coherence_10_mean,word_coherence_10_var,word_coherence_variability_2_mean,word_coherence_variability_2_var,word_coherence_variability_3_mean,word_coherence_variability_3_var,word_coherence_variability_4_mean,word_coherence_variability_4_var,word_coherence_variability_5_mean,word_coherence_variability_5_var,word_coherence_variability_6_mean,word_coherence_variability_6_var,word_coherence_variability_7_mean,word_coherence_variability_7_var,word_coherence_variability_8_mean,word_coherence_variability_8_var,word_coherence_variability_9_mean,word_coherence_variability_9_var,word_coherence_variability_10_mean,word_coherence_variability_10_var,num_turns,num_one_word_turns,mean_turn_length_minutes,mean_turn_length_words,mean_pre_turn_pause,speaker_percentage,first_order_sentence_tangeniality_mean,first_order_sentence_tangeniality_var,second_order_sentence_tangeniality_mean,second_order_sentence_tangeniality_var,turn_to_turn_tangeniality_mean,turn_to_turn_tangeniality_var,turn_to_turn_tangeniality_slope,semantic_perplexity_mean,semantic_perplexity_var,semantic_perplexity_5_mean,semantic_perplexity_5_var,semantic_perplexity_11_mean,semantic_perplexity_11_var,semantic_perplexity_15_mean,semantic_perplexity_15_var,num_interrupts,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader,first_person_sentiment_overall_vader
0,10.692533,5.850883,619,105.795991,159.292187,0.199697,0.653059,54.719337,0.304646,0.225927,0.469427,0.078719,0.271,0.042,0.687,0.9995,0.970943,0.923676,0.812249,0.70691,0.598333,6.447535,0.196629,0.383436,28.383173,1.035133,24.53517,1.418885,0.0,0.574664,0.004783,0.662479,0.001592,0.613879,0.001353,0.572357,0.006101,0.568598,0.007325,0.573143,0.00685,0.563139,0.00661,0.566036,0.005412,0.592895,0.01161,0.592204,0.013491,0.572399,0.008054,0.568776,0.004641,127,25,0.04607,4.874016,2.113921,26.730803,NaN,NaN,NaN,NaN,0.468444,0.010288,-0.000748,839.765021,1.684831e+07,18632.921568,1.285061e+10,2702.609941,3.832120e+08,1155.273834,3.421040e+07,0,22.35711,0.275492,22.439984


In [18]:
from openwillis.speech.util.speech import coherence
import openwillis.speech as ows
coherence.COHERENCE_BACKEND = "gemma"

words_ukr_old, turns_ukr_old, summary_ulr_old = ows.speech_characteristics(
    json_conf=ukr_data,
    option="coherence",
    language="ua",
    speaker_label="participant",
    min_coherence_turn_length=2,
    whisper_turn_mode="segment",
)

INFO:pymorphy3.opencorpora_dict.wrapper:Loading dictionaries from /Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/pymorphy3_dicts_uk/data
INFO:pymorphy3.opencorpora_dict.wrapper:format: 2.4, revision: 1, updated: 2022-09-13T18:45:24.998984


Try edit function....


/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: 'ь'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: '—'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: 'ь'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/nltk/tokenize/sonority_sequencing.py:102: UserWarning: Character not defined in sonority_hierarchy, assigning as vowel: '—'
  warnings.warn(
/Users/pelmeshek1706/Desktop/projects/fi

In [19]:
display(summary_ukr)
display(summary_ulr_old)

,file_length,speech_length_minutes,speech_length_words,words_per_min,syllables_per_min,mean_pre_word_pause,mean_pause_variability,speech_percentage,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,prop_verb_past,prop_function_words,first_person_sentiment_positive,first_person_sentiment_negative,first_person_sentiment_overall,word_repeat_percentage,phrase_repeat_percentage,word_coherence_mean,word_coherence_var,word_coherence_5_mean,word_coherence_5_var,word_coherence_10_mean,word_coherence_10_var,word_coherence_variability_2_mean,word_coherence_variability_2_var,word_coherence_variability_3_mean,word_coherence_variability_3_var,word_coherence_variability_4_mean,word_coherence_variability_4_var,word_coherence_variability_5_mean,word_coherence_variability_5_var,word_coherence_variability_6_mean,word_coherence_variability_6_var,word_coherence_variability_7_mean,word_coherence_variability_7_var,word_coherence_variability_8_mean,word_coherence_variability_8_var,word_coherence_variability_9_mean,word_coherence_variability_9_var,word_coherence_variability_10_mean,word_coherence_variability_10_var,num_turns,num_one_word_turns,mean_turn_length_minutes,mean_turn_length_words,mean_pre_turn_pause,speaker_percentage,first_order_sentence_tangeniality_mean,first_order_sentence_tangeniality_var,second_order_sentence_tangeniality_mean,second_order_sentence_tangeniality_var,turn_to_turn_tangeniality_mean,turn_to_turn_tangeniality_var,turn_to_turn_tangeniality_slope,semantic_perplexity_mean,semantic_perplexity_var,semantic_perplexity_5_mean,semantic_perplexity_5_var,semantic_perplexity_11_mean,semantic_perplexity_11_var,semantic_perplexity_15_mean,semantic_perplexity_15_var,num_interrupts,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader,first_person_sentiment_overall_vader
0,10.692533,5.850883,520,88.875469,191.253173,0.0,0.0,54.719337,0.280084,0.238961,0.480955,0.041122,0.308,0.041,0.651,0.9995,0.977143,0.936538,0.854337,0.776708,0.695395,5.865522,0.288136,0.356061,26.317564,1.361495,20.904131,0.405512,0.0,0.606044,0.008957,0.697155,0.003225,0.668208,0.001623,0.599774,0.010808,0.600044,0.010539,0.610938,0.012413,0.604197,0.009801,0.611355,0.008289,0.585886,0.010951,0.624852,0.016538,0.600263,0.006922,0.584019,0.013484,127,26,0.04607,4.094488,2.113921,26.730803,NaN,NaN,NaN,NaN,0.440573,0.011523,0.000203,712.684793,6.295328e+06,269890.043828,3.353436e+11,4372.1993,3.065547e+07,1463.982299,7.519573e+06,0,25.648716,0.28315,25.703991


,file_length,speech_length_minutes,speech_length_words,words_per_min,syllables_per_min,mean_pre_word_pause,mean_pause_variability,speech_percentage,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,prop_verb_past,prop_function_words,first_person_sentiment_positive,first_person_sentiment_negative,first_person_sentiment_overall,word_repeat_percentage,phrase_repeat_percentage,word_coherence_mean,word_coherence_var,word_coherence_5_mean,word_coherence_5_var,word_coherence_10_mean,word_coherence_10_var,word_coherence_variability_2_mean,word_coherence_variability_2_var,word_coherence_variability_3_mean,word_coherence_variability_3_var,word_coherence_variability_4_mean,word_coherence_variability_4_var,word_coherence_variability_5_mean,word_coherence_variability_5_var,word_coherence_variability_6_mean,word_coherence_variability_6_var,word_coherence_variability_7_mean,word_coherence_variability_7_var,word_coherence_variability_8_mean,word_coherence_variability_8_var,word_coherence_variability_9_mean,word_coherence_variability_9_var,word_coherence_variability_10_mean,word_coherence_variability_10_var,num_turns,num_one_word_turns,mean_turn_length_minutes,mean_turn_length_words,mean_pre_turn_pause,speaker_percentage,first_order_sentence_tangeniality_mean,first_order_sentence_tangeniality_var,second_order_sentence_tangeniality_mean,second_order_sentence_tangeniality_var,turn_to_turn_tangeniality_mean,turn_to_turn_tangeniality_var,turn_to_turn_tangeniality_slope,semantic_perplexity_mean,semantic_perplexity_var,semantic_perplexity_5_mean,semantic_perplexity_5_var,semantic_perplexity_11_mean,semantic_perplexity_11_var,semantic_perplexity_15_mean,semantic_perplexity_15_var,num_interrupts,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader,first_person_sentiment_overall_vader
0,10.692533,5.850883,520,88.875469,191.253173,0.0,0.0,54.719337,0.280084,0.238961,0.480955,0.041122,0.308,0.041,0.651,0.9995,0.977143,0.936538,0.854337,0.776708,0.695395,5.865522,0.288136,0.356061,26.317564,1.361495,20.904131,0.405512,0.0,0.606044,0.008957,0.697155,0.003225,0.668208,0.001623,0.599774,0.010808,0.600044,0.010539,0.610938,0.012413,0.604197,0.009801,0.611355,0.008289,0.585886,0.010951,0.624852,0.016538,0.600263,0.006922,0.584019,0.013484,127,26,0.04607,4.094488,2.113921,26.730803,NaN,NaN,NaN,NaN,0.440573,0.011523,0.000203,712.684793,6.295328e+06,269890.043828,3.353436e+11,4372.1993,3.065547e+07,1463.982299,7.519573e+06,0,25.648716,0.28315,25.703991
